# Present Your Solution

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import set_config
set_config(transform_output="pandas")

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Copied from the Preparation notebook (needed to load preprocessor.pkl)
class EarlySpringFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, add_deltas=True, add_pre_march_trend=True, combine_oni=True, drop_redundant=True, use_year=False,
                 delta_features=("soil_temp_mean", "farm_avg_temp_mean", "om_temp_mean"),
                 redundant_features=("soil_thawed_days_jan", "soil_thawed_days_feb", "farm_freezing_days_feb")):
        self.add_deltas = add_deltas
        self.add_pre_march_trend = add_pre_march_trend
        self.combine_oni = combine_oni
        self.drop_redundant = drop_redundant
        self.use_year = use_year
        self.delta_features = delta_features
        self.redundant_features = redundant_features

    def fit(self, X, y=None):
        self.feature_names_out_ = self._engineer(X).columns.to_numpy()
        return self

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_

    def transform(self, X):
        return self._engineer(X)

    def _engineer(self, X):
        data = X.drop(columns="early_spring", errors="ignore")
        data["phil_pred"] = data["phil_pred"].astype(int)

        if self.add_deltas:
            for name in self.delta_features:
                data[f"{name}_delta"] = data[f"{name}_feb"] - data[f"{name}_jan"]

        if self.add_pre_march_trend:
            for prefix in ["farm_avg_temp", "soil_mean_soil_temp", "om_temperature_2m_mean"]:
                d1 = f"{prefix}_d_minus_1"
                d7 = f"{prefix}_d_minus_7"
                if d1 in data.columns and d7 in data.columns:
                    data[f"{prefix}_7d_trend"] = data[d1] - data[d7]

        if self.combine_oni:
            data["oni_winter_avg"] = data[["oni_prev_nov", "oni_prev_dec"]].mean(axis=1)
            data = data.drop(columns=["oni_prev_nov", "oni_prev_dec"])

        if self.drop_redundant:
            data = data.drop(columns=list(self.redundant_features), errors="ignore")

        if self.use_year:
            data["year"] = data.index.to_numpy()

        return data

In [ ]:
with open("model.pkl", "rb") as f:
    model = pickle.load(f)

# The test set has not been used for anything yet - this is the final, honest check
test_set = pd.read_parquet("test_set.parquet")
X_test = test_set.drop(columns="early_spring")
y_test = test_set["early_spring"].astype(int)

model_preds = model.predict(X_test)
phil_preds = X_test["phil_pred"].astype(int)

In [ ]:
# Model vs Phil on the test set
scores = pd.DataFrame({
    name: {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
    }
    for name, preds in [("Model", model_preds), ("Phil", phil_preds)]
}).T
scores.round(3)

In [ ]:
scores.T.plot.bar(figsize=(8, 5), rot=0)
plt.ylim(0, 1)
plt.ylabel("Score (test set)")
plt.title("Model vs Punxsutawney Phil")
plt.show()

In [ ]:
# Rows = actual, columns = predicted (0 = late spring, 1 = early spring)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, preds) in zip(axes, [("Model", model_preds), ("Phil", phil_preds)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, preds), display_labels=["Late", "Early"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()

In [ ]:
# Year-by-year predictions on the test set
pd.DataFrame({
    "actual": y_test.map({1: "Early", 0: "Late"}),
    "phil": phil_preds.map({1: "Early", 0: "Late"}),
    "model": pd.Series(model_preds, index=y_test.index).map({1: "Early", 0: "Late"}),
}).sort_index()